# 03 Features

builds the feature table that xgboost and the deep learning models will share. every feature is checked to make sure it only uses information known on the monday the forecast is made.

## Set-up (Imports and Loading Data)

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

import sys
sys.path.append('../src')

from backtest import split_data_by_cutoff
from baselines import add_all_baselines
from features import build_features, FEATURE_COLUMNS, check_no_leakage
from load_clean_data import load_clean_data

In [2]:
df = load_clean_data()

## Building the Features

features are built once for the whole dataset. every feature looks back at least 7 days from its own row, so one table is valid for any 7 day forecast made on a monday cutoff. no need to rebuild it per cutoff.

In [3]:
df_with_features = build_features(df)

assert len(df_with_features) == len(df), "Feature engineering should not change the number of rows in the dataset."
assert set(FEATURE_COLUMNS) <= set(df_with_features.columns), "The featured dataframe should contain all the feature columns defined in FEATURE_COLUMNS."
assert not {'Sales', 'Customers', 'Open'} & set(FEATURE_COLUMNS), "Feature columns should not include the target variable or other non-feature columns."

what the models are allowed to use:
- calendar: day of week, month, day of month, week of year, day of year
- promo and holidays: Promo, SchoolHoliday, IsStateHoliday, StateHolidayType (promos and holiday dates are known in advance)
- store info: StoreType, Assortment, CompetitionDistance, Promo2
- CompetitionMonths: months since the nearest competitor opened (0 if it has not opened yet)
- sales history: sales 7, 14 and 28 days earlier, and the same weekday average over the last 4 and 8 weeks

In [4]:
print(df_with_features[FEATURE_COLUMNS].shape)
print(df_with_features[FEATURE_COLUMNS].dtypes)

(1017209, 19)
DayOfWeek                 int64
Promo                     int64
SchoolHoliday             int64
IsStateHoliday            int64
StateHolidayType       category
Month                     int32
DayOfMonth                int32
WeekOfYear                int64
DayOfYear                 int32
StoreType              category
Assortment             category
CompetitionDistance     float64
CompetitionMonths       float64
Promo2                    int64
SalesLag7               float64
SalesLag14              float64
SalesLag28              float64
SalesLagMean4w          float64
SalesLagMean8w          float64
dtype: object


left out on purpose:
- Customers: not known on monday, and it is closely tied to sales
- Open: some closures are unplanned, so the models learn the sunday zeros from DayOfWeek, like the baselines do
- Year: the data only covers 2013 to 2015 and the sales history features already carry the level
- Store id: the sales history features already carry each store's level

In [5]:
print(df_with_features[FEATURE_COLUMNS].isnull().sum())

DayOfWeek                   0
Promo                       0
SchoolHoliday               0
IsStateHoliday              0
StateHolidayType            0
Month                       0
DayOfMonth                  0
WeekOfYear                  0
DayOfYear                   0
StoreType                   0
Assortment                  0
CompetitionDistance         0
CompetitionMonths      323348
Promo2                      0
SalesLag7                9065
SalesLag14              18130
SalesLag28              36260
SalesLagMean4w          36260
SalesLagMean8w          72520
dtype: int64


missing values are expected and are not a problem:
- the lag and mean columns are NaN for the first 7 to 56 days of each store, and again after the gap in the 180 gap stores. e.g. SalesLag7 is missing for (1,115 + 180) x 7 = 9,065 rows
- CompetitionMonths is NaN for 354 stores that have no competitor opening date (323,348 rows)

xgboost handles NaN directly. the deep learning models will need a fill value and a flag column for these later.

## Baseline Match

check: for the cutoff 2015-02-02, the 4 week average feature should equal the 28 day average baseline from notebook 02, since they are the same calculation.

In [6]:
cutoff = '2015-02-02'

train, test = split_data_by_cutoff(df, cutoff, 7)
baseline_rows = add_all_baselines(train, test, cutoff)[['Store', 'Date', 'avg_28d_baseline']]

feature_rows = df_with_features[
    (df_with_features['Date'] >= cutoff) & (df_with_features['Date'] < pd.Timestamp(cutoff) + pd.Timedelta(days=7))
][['Store', 'Date', 'SalesLagMean4w']]

compare = baseline_rows.merge(feature_rows, on=['Store', 'Date'], how='inner', validate='one_to_one')
compare = compare.dropna()

print(len(baseline_rows), len(feature_rows), len(compare))

assert len(compare) == len(baseline_rows), "rows were lost in the merge or to NaN"
assert np.allclose(compare['avg_28d_baseline'], compare['SalesLagMean4w']), "feature and baseline disagree"

7805 7805 7805


all 7,805 rows match, gap stores included.

## Leakage Test

check for leakage: remove all sales from the cutoff onwards and rebuild the features. the features for the test week should not change. as a control, removing the week before the cutoff should change them, otherwise the test would prove nothing.

In [7]:
for cutoff in ['2015-02-02', '2015-04-06', '2015-07-20']:
    check_no_leakage(df, df_with_features, cutoff)

2015-02-02: no leakage, control passed
2015-04-06: no leakage, control passed
2015-07-20: no leakage, control passed


no leakage at 2015-02-02, 2015-04-06 and 2015-07-20, and the control passed each time.